# FastAPI


## FastAPI 是什么

FastAPI 是一个用 Python 构建 Web API 的框架。  
异步性能高，有类型校验，在线接口测试  
它读取函数签名和类型注解，把同一份声明同时用于接收请求、校验数据、调用业务代码、序列化响应和生成 API 文档。

FastAPI 本身不是负责监听端口的 Web 服务器。一个典型组合由三层构成：

| 组件 | 负责什么 | 关系 |
| --- | --- | --- |
| FastAPI | 路由、依赖注入、异常处理、OpenAPI 集成 | 开发者直接使用的框架 |
| Starlette | ASGI、请求与响应、中间件、WebSocket | FastAPI 的 Web 基础 |
| Pydantic | 类型驱动的数据校验、转换、序列化与 JSON Schema | FastAPI 的数据基础 |
| Uvicorn | 监听端口并把 HTTP/WebSocket 请求交给应用 | 运行 FastAPI 的 ASGI 服务器 |

因此，`app = FastAPI()` 创建的是 ASGI 应用对象，Uvicorn 才是启动进程并监听端口的程序。

与常见 Python Web 框架相比：

| 框架 | 主要定位 | 特点 |
| --- | --- | --- |
| Flask | 轻量 Web 框架 | 核心很小，校验、文档等能力通常自行选扩展 |
| Django | 全功能 Web 框架 | 自带 ORM、后台管理、模板、认证等完整体系 |
| FastAPI | API 优先的 Web 框架 | 类型注解贯穿校验、序列化和文档，原生支持 ASGI |

FastAPI 适合 JSON API、微服务、AI 服务接口和需要异步 I/O 的后端。它不自带 ORM、任务队列或前端模板体系，这些能力需要按项目选择。

前后端分离：前端的（界面效果）和后端的（业务逻辑，返回数据）由两个服务端实现

## 一次请求如何经过 FastAPI

理解请求链路，才能判断错误究竟发生在服务器、框架、数据校验还是业务代码中。

```mermaid
flowchart LR
    A["客户端"] --> B["Uvicorn<br/>接收 HTTP 请求"]
    B --> C["中间件"]
    C --> D["路由匹配"]
    D --> E["解析并校验参数"]
    E --> F["解析依赖项"]
    F --> G["路径操作函数"]
    G --> H["响应校验与序列化"]
    H --> B
    B --> A
```

如果路由不存在，通常返回 `404`；输入不符合类型或约束，FastAPI 默认返回 `422`；业务代码主动拒绝请求，可以抛出 `HTTPException`；未处理异常通常成为 `500`。

### ASGI 与 WSGI

| | WSGI | ASGI |
| --- | --- | --- |
| 主要模型 | 同步请求与响应 | 同步、异步、长连接 |
| 常见场景 | 传统 HTTP Web 应用 | HTTP、WebSocket、流式响应 |
| 常见服务器 | Gunicorn、uWSGI | Uvicorn、Hypercorn、Daphne |

ASGI 是服务器与 Python 应用之间的接口规范。FastAPI 支持异步，不代表每段代码都会自动并发；只有把等待型操作交给可等待的异步库，并正确使用 `await`，事件循环才能在等待期间处理其他请求。

## 安装、创建与运行应用

安装带标准运行组件的 FastAPI：

```bash
uv add "fastapi[standard]"
```

创建 `main.py`：

```python
from fastapi import FastAPI

app = FastAPI(title="示例 API")


@app.get("/health")
async def health_check():
    return {"status": "ok"}
```

开发时启动自动重载服务器：

```bash
fastapi dev main.py
```

也可以直接使用 Uvicorn：

```bash
uvicorn main:app --reload
```

`main:app` 表示从 Python 模块 `main` 中导入变量 `app`。`--reload` 会监视文件并重启进程，只用于开发环境。生产环境可以使用 `fastapi run main.py`，并由部署环境管理进程、端口和副本数。

启动后常用地址：

| 地址 | 用途 |
| --- | --- |
| `http://127.0.0.1:8000/health` | 调用接口 |
| `http://127.0.0.1:8000/docs` | Swagger UI 交互文档 |
| `http://127.0.0.1:8000/redoc` | ReDoc 阅读文档 |
| `http://127.0.0.1:8000/openapi.json` | OpenAPI 机器可读描述 |

## 路由与路径操作

路由决定某种 HTTP 方法和 URL 路径由哪个函数处理。FastAPI 把这些处理函数称为路径操作函数。

```python
from fastapi import FastAPI, status

app = FastAPI()


@app.get("/items", tags=["items"], summary="查询商品列表")
async def list_items():
    return []


@app.post("/items", status_code=status.HTTP_201_CREATED)
async def create_item():
    return {"id": 1}
```

装饰器中的 `get`、`post`、`put`、`patch`、`delete` 对应 HTTP 方法。`tags` 用来给文档分组，`summary` 用来描述操作；函数名最好表达业务动作，因为它也会影响自动生成的 OpenAPI 操作标识。

路由按声明顺序匹配。固定路径应写在动态路径前面：

```python
@app.get("/users/me")
async def read_current_user():
    return {"id": "current"}


@app.get("/users/{user_id}")
async def read_user(user_id: str):
    return {"id": user_id}
```

如果先声明 `/users/{user_id}`，请求 `/users/me` 可能先被当作 `user_id="me"` 处理。

## 路径、查询、请求头与 Cookie 参数

FastAPI 会根据参数位置和类型推断数据来源，也可以用 `Path`、`Query`、`Header`、`Cookie` 明确声明约束。

```python
from typing import Annotated

from fastapi import Cookie, FastAPI, Header, Path, Query

app = FastAPI()


@app.get("/items/{item_id}")
async def read_item(
    item_id: Annotated[int, Path(gt=0)],
    q: Annotated[str | None, Query(min_length=2, max_length=50)] = None,
    user_agent: Annotated[str | None, Header()] = None,
    session_id: Annotated[str | None, Cookie()] = None,
):
    return {
        "item_id": item_id,
        "q": q,
        "user_agent": user_agent,
        "session_id": session_id,
    }
```

请求 `/items/12?q=keyboard` 时，`item_id` 来自路径，`q` 来自查询字符串。字符串 `12` 会被转换为整数；转换失败或不满足 `gt=0` 时，请求不会进入函数，FastAPI 直接返回校验错误。

| 声明 | 数据来源 | 备注 |
| --- | --- | --- |
| 路由中同名参数 | 路径 | 例如 `{item_id}` |
| 简单类型参数 | 查询字符串 | 不在路径中时默认如此 |
| `Header()` | 请求头 | Python 的下划线默认转换为连字符，`user_agent` 对应 `User-Agent` |
| `Cookie()` | Cookie | 需要显式声明 |
| Pydantic 模型 | 请求体 | 默认按 JSON 解析 |

推荐用 `Annotated` 把 Python 类型和 FastAPI 元数据放在一起，既保留类型检查，也能集中表达校验规则。

## 请求体与 Pydantic 模型

请求体通常是结构化 JSON。只接收一个无约束的 `dict`，框架无法知道哪些字段必填、类型是否正确，也无法生成准确文档；Pydantic 模型把这份数据契约写进代码。

```python
from typing import Annotated

from fastapi import FastAPI, Query
from pydantic import BaseModel, Field

app = FastAPI()


class ItemCreate(BaseModel):
    name: str = Field(min_length=1, max_length=100)
    price: float = Field(gt=0)
    description: str | None = Field(default=None, max_length=500)
    tags: list[str] = Field(default_factory=list)


@app.post("/shops/{shop_id}/items")
async def create_item(
    shop_id: int,
    item: ItemCreate,
    notify: Annotated[bool, Query()] = False,
):
    return {"shop_id": shop_id, "notify": notify, "item": item}
```

请求体示例：

```json
{
  "name": "机械键盘",
  "price": 499.0,
  "tags": ["keyboard", "hardware"]
}
```

Pydantic 会完成四件事：检查必填字段、转换兼容类型、执行字段约束、生成 JSON Schema。模型实例可通过 `item.name` 访问字段，通过 `item.model_dump()` 转成字典。

输入模型应按业务场景拆分，例如 `ItemCreate`、`ItemUpdate`。更新模型通常把字段设为可选，再用 `model_dump(exclude_unset=True)` 区分“客户端没有发送”与“客户端明确发送了空值”。

## 响应模型、序列化与状态码

返回值不仅要能转成 JSON，还要保证对外字段稳定且不会泄露内部数据。`response_model` 是响应契约：FastAPI 会按它校验、过滤和序列化返回值。

```python
from fastapi import FastAPI, status
from pydantic import BaseModel, ConfigDict

app = FastAPI()


class UserCreate(BaseModel):
    username: str
    password: str


class UserRead(BaseModel):
    model_config = ConfigDict(from_attributes=True)

    id: int
    username: str


@app.post(
    "/users",
    response_model=UserRead,
    status_code=status.HTTP_201_CREATED,
)
async def create_user(payload: UserCreate):
    return {
        "id": 1,
        "username": payload.username,
        "password": "内部字段不会出现在响应中",
    }
```

即使函数返回了 `password`，响应模型也只保留 `id` 和 `username`。`from_attributes=True` 允许模型从 ORM 对象等属性对象读取字段。

常见状态码：

| 状态码 | 常见用途 |
| --- | --- |
| `200 OK` | 查询、更新成功 |
| `201 Created` | 创建成功 |
| `204 No Content` | 删除成功且没有响应体 |
| `400 Bad Request` | 请求在业务语义上无效 |
| `401 Unauthorized` | 未提供有效身份凭证 |
| `403 Forbidden` | 身份有效但无权限 |
| `404 Not Found` | 资源不存在 |
| `409 Conflict` | 唯一键冲突、状态冲突 |
| `422 Unprocessable Entity` | FastAPI 默认的输入校验失败 |

不要把数据库模型直接同时当作创建输入和公开响应。输入、持久化、输出承担不同职责，分开建模才能控制可写字段和敏感字段。

## 错误处理

可预期的业务失败应该转成明确的 HTTP 响应，而不是让它变成 `500`。最直接的方式是抛出 `HTTPException`：

```python
from fastapi import FastAPI, HTTPException, status

app = FastAPI()


@app.get("/items/{item_id}")
async def read_item(item_id: int):
    item = None
    if item is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="商品不存在",
        )
    return item
```

业务较大时，业务层可以抛领域异常，再由框架边界统一转换，避免业务代码到处依赖 HTTP：

```python
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

app = FastAPI()


class StockConflictError(Exception):
    pass


@app.exception_handler(StockConflictError)
async def handle_stock_conflict(
    request: Request, exc: StockConflictError
) -> JSONResponse:
    return JSONResponse(
        status_code=409,
        content={"detail": str(exc)},
    )
```

框架会自动处理参数校验错误。除非客户端契约明确要求统一错误格式，否则不必急着覆盖默认处理器。记录服务端异常时保留完整日志，但不要把堆栈、SQL、密钥或内部路径返回给客户端。

## 依赖注入

认证、数据库会话、配置读取等逻辑会被许多接口复用。直接在每个函数中重复创建和清理资源，容易遗漏；FastAPI 用 `Depends` 声明“运行这个接口前需要什么”。

```python
from typing import Annotated

from fastapi import Depends, FastAPI, Header, HTTPException

app = FastAPI()


async def require_api_key(
    x_api_key: Annotated[str, Header()],
) -> str:
    if x_api_key != "development-key":
        raise HTTPException(status_code=401, detail="无效的 API Key")
    return x_api_key


@app.get("/private")
async def private_data(
    api_key: Annotated[str, Depends(require_api_key)],
):
    return {"authenticated": True}
```

依赖函数也能依赖其他依赖，形成依赖树。FastAPI 会先解析树，再调用路径操作函数；同一个请求中，相同依赖默认只执行一次并缓存结果。

需要清理的资源使用 `yield`，`yield` 前获取资源，`yield` 后释放资源：

```python
from collections.abc import AsyncIterator

import httpx


async def get_http_client() -> AsyncIterator[httpx.AsyncClient]:
    async with httpx.AsyncClient(timeout=5.0) as client:
        yield client
```

数据库会话也常用同一模式：创建会话、`yield` 给接口、最后关闭。应写 `Depends(get_http_client)`，传入函数本身；写成 `Depends(get_http_client())` 会在声明时调用它，是常见错误。

## 认证与授权依赖

认证回答“你是谁”，授权回答“你能做什么”。FastAPI 的安全工具负责从请求中提取凭证并写入 OpenAPI，但不会自动验证令牌或设计权限规则。

```python
from typing import Annotated

from fastapi import Depends, FastAPI, HTTPException, status
from fastapi.security import OAuth2PasswordBearer
from pydantic import BaseModel

app = FastAPI()
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/auth/token")


class User(BaseModel):
    id: int
    role: str


def verify_token_and_load_user(token: str) -> User | None:
    # 实际项目在这里校验签名、过期时间、签发者和受众。
    return User(id=1, role="admin") if token == "demo" else None


async def get_current_user(
    token: Annotated[str, Depends(oauth2_scheme)],
) -> User:
    user = verify_token_and_load_user(token)
    if user is None:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="无效的访问令牌",
            headers={"WWW-Authenticate": "Bearer"},
        )
    return user


def require_admin(
    user: Annotated[User, Depends(get_current_user)],
) -> User:
    if user.role != "admin":
        raise HTTPException(status_code=403, detail="权限不足")
    return user


@app.delete("/admin/items/{item_id}")
async def delete_item(
    item_id: int,
    admin: Annotated[User, Depends(require_admin)],
):
    return {"deleted": item_id, "operator": admin.id}
```

`OAuth2PasswordBearer` 只读取 `Authorization: Bearer ...` 中的令牌，并让文档显示认证入口。真正的密码校验、令牌签发、签名验证和权限判断仍由应用实现。

## APIRouter 与项目结构

所有接口都写在 `main.py` 中时，路由、业务规则和数据库代码会迅速混在一起。`APIRouter` 可以按业务领域拆分路由，再由主应用统一挂载。

一个中型项目可以从下面的结构开始：

```
app/
├── main.py
├── api/
│   ├── dependencies.py
│   └── routes/
│       ├── items.py
│       └── users.py
├── schemas/
│   ├── item.py
│   └── user.py
├── services/
│   └── item_service.py
├── repositories/
│   └── item_repository.py
├── db.py
└── settings.py
tests/
```

`app/api/routes/items.py`：

```python
from fastapi import APIRouter

router = APIRouter(prefix="/items", tags=["items"])


@router.get("")
async def list_items():
    return []


@router.get("/{item_id}")
async def read_item(item_id: int):
    return {"id": item_id}
```

`app/main.py`：

```python
from fastapi import FastAPI

from app.api.routes.items import router as items_router

app = FastAPI()
app.include_router(items_router, prefix="/api/v1")
```

路由层负责 HTTP 输入输出，服务层负责业务规则，仓储层负责持久化访问。小项目不必为了形式建齐所有层；当一个路由函数开始混合校验、事务、外部调用和复杂分支时，再把相应职责移出去。

## 中间件与 CORS

中间件包围每个请求，适合处理请求 ID、访问日志、耗时统计、压缩等全局横切逻辑。它不适合承载具体业务规则。

```python
from time import perf_counter

from fastapi import FastAPI, Request

app = FastAPI()


@app.middleware("http")
async def add_process_time(request: Request, call_next):
    started = perf_counter()
    response = await call_next(request)
    response.headers["X-Process-Time"] = f"{perf_counter() - started:.4f}"
    return response
```

浏览器前端与 API 的源不同，例如前端是 `http://localhost:5173`、后端是 `http://localhost:8000`，浏览器会执行 CORS 规则。后端需要明确允许可信前端源：

```python
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)
```

CORS 是浏览器的跨域读取限制，不是身份认证或服务器防火墙。使用 Cookie 或认证头时应列出明确来源，不要为了省事在生产环境开放任意源。

## 应用生命周期与共享资源

数据库连接池、HTTP 客户端和机器学习模型不应该在每次请求时重新创建，也需要在进程退出时正确关闭。FastAPI 推荐用 `lifespan` 管理应用级资源。

```python
from contextlib import asynccontextmanager

import httpx
from fastapi import FastAPI, Request


@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.http_client = httpx.AsyncClient(timeout=5.0)
    yield
    await app.state.http_client.aclose()


app = FastAPI(lifespan=lifespan)


@app.get("/upstream")
async def call_upstream(request: Request):
    client: httpx.AsyncClient = request.app.state.http_client
    response = await client.get("https://example.com/api")
    return response.json()
```

`yield` 之前的代码在应用开始接收请求前执行，之后的代码在应用停止时执行。每个服务器进程都有自己的内存和生命周期；启动多个 worker 后，内存缓存和全局变量不会自动共享。

## `async def`、`def` 与并发

异步的价值主要来自等待期间让出执行权。网络请求、异步数据库查询等 I/O 往往大部分时间在等待，适合异步；压缩、图像处理、模型推理等 CPU 密集工作不会因为加上 `async` 而变快。

| 工作类型 | 路径操作写法 | 原因 |
| --- | --- | --- |
| 使用异步库等待网络、数据库、文件 | `async def` 并 `await` | 等待时事件循环可处理其他任务 |
| 只能调用阻塞式 I/O 库 | 普通 `def`，或显式放入线程池 | FastAPI 会在线程池执行普通路径操作 |
| CPU 密集计算 | 独立进程、任务队列或专用计算服务 | 避免阻塞事件循环和 Web worker |

正确的异步等待：

```python
import asyncio

from fastapi import FastAPI

app = FastAPI()


@app.get("/wait")
async def wait_briefly():
    await asyncio.sleep(0.1)
    return {"done": True}
```

错误做法是在 `async def` 中调用 `time.sleep()`、同步 HTTP 客户端或长时间 CPU 计算，它们会堵住事件循环。也不要无条件把所有函数改成异步；普通计算函数没有等待点，保持普通 `def` 更清楚。

## 文件、表单与后台任务

JSON 之外，FastAPI 也能接收 `multipart/form-data`。上传大文件时使用 `UploadFile`，它提供文件名、内容类型和文件对象，并会在超过内存阈值后使用临时文件。

```python
from typing import Annotated

from fastapi import BackgroundTasks, FastAPI, File, Form, UploadFile

app = FastAPI()


def write_audit_log(filename: str) -> None:
    print(f"uploaded: {filename}")


@app.post("/documents")
async def upload_document(
    background_tasks: BackgroundTasks,
    title: Annotated[str, Form()],
    file: Annotated[UploadFile, File()],
):
    content = await file.read()
    background_tasks.add_task(write_audit_log, file.filename or "unknown")
    return {
        "title": title,
        "filename": file.filename,
        "size": len(content),
    }
```

`BackgroundTasks` 会在响应发出后、当前应用进程内执行任务，适合短小且失败后可以接受的工作，例如写审计记录。发送账单、视频转码等必须重试或耗时较长的任务，应交给持久化任务队列；进程崩溃时，内存中的后台任务可能丢失。

真实文件上传还要限制大小、校验实际内容、使用安全生成的存储名，并避免直接信任客户端提供的文件名和 `Content-Type`。

## WebSocket 与流式响应

普通 HTTP 是一次请求对应一次响应；实时聊天、状态推送等场景需要长连接。FastAPI 基于 ASGI 支持 WebSocket：

```python
from fastapi import FastAPI, WebSocket, WebSocketDisconnect

app = FastAPI()


@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()
    try:
        while True:
            message = await websocket.receive_text()
            await websocket.send_json({"echo": message})
    except WebSocketDisconnect:
        pass
```

需要逐块返回普通 HTTP 内容时，可以使用 `StreamingResponse`：

```python
from collections.abc import AsyncIterator

from fastapi.responses import StreamingResponse


async def generate_events() -> AsyncIterator[str]:
    yield "data: ready\n\n"


@app.get("/events")
async def events():
    return StreamingResponse(
        generate_events(),
        media_type="text/event-stream",
    )
```

单进程内用列表保存 WebSocket 连接只能服务当前进程。多 worker、多机器广播时，需要 Redis Pub/Sub 等外部消息系统协调连接所在的各个进程。

## OpenAPI 与自动文档

FastAPI 会把路由、参数、Pydantic 模型、状态码和安全声明组合成 OpenAPI 文档。Swagger UI 和 ReDoc 都是这份 `/openapi.json` 的可视化界面，前端还可以据此生成类型和 API 客户端。

```python
from fastapi import FastAPI

tags_metadata = [
    {"name": "items", "description": "商品查询与维护"},
    {"name": "users", "description": "用户与身份信息"},
]

app = FastAPI(
    title="商城 API",
    version="1.0.0",
    description="商城后端的公开接口",
    openapi_tags=tags_metadata,
)
```

自动文档准确与否取决于代码契约是否准确。使用模糊的 `dict`、遗漏响应模型或把所有失败都写成 `200`，生成的文档也会失真。对外接口还应通过 `responses` 补充重要错误响应。

关闭 `/docs` 不能代替认证和授权。文档是否公开是产品与安全决策，真正的接口仍必须独立执行访问控制。

## 一个完整的 CRUD 示例

下面的单文件示例把输入模型、部分更新、响应模型、状态码和异常处理连在一起，可以直接运行。内存字典只用于演示，进程重启后数据会消失，也不能替代真实数据库。

```python
from fastapi import FastAPI, HTTPException, Response, status
from pydantic import BaseModel, Field

app = FastAPI(title="Inventory API")


class ItemCreate(BaseModel):
    name: str = Field(min_length=1, max_length=100)
    price: float = Field(gt=0)


class ItemUpdate(BaseModel):
    name: str | None = Field(default=None, min_length=1, max_length=100)
    price: float | None = Field(default=None, gt=0)


class ItemRead(ItemCreate):
    id: int


items: dict[int, ItemRead] = {}


def get_item_or_404(item_id: int) -> ItemRead:
    item = items.get(item_id)
    if item is None:
        raise HTTPException(status_code=404, detail="商品不存在")
    return item


@app.post(
    "/items",
    response_model=ItemRead,
    status_code=status.HTTP_201_CREATED,
)
async def create_item(payload: ItemCreate):
    item_id = max(items, default=0) + 1
    item = ItemRead(id=item_id, **payload.model_dump())
    items[item_id] = item
    return item


@app.get("/items", response_model=list[ItemRead])
async def list_items(skip: int = 0, limit: int = 20):
    return list(items.values())[skip : skip + limit]


@app.get("/items/{item_id}", response_model=ItemRead)
async def read_item(item_id: int):
    return get_item_or_404(item_id)


@app.patch("/items/{item_id}", response_model=ItemRead)
async def update_item(item_id: int, payload: ItemUpdate):
    stored = get_item_or_404(item_id)
    changes = payload.model_dump(exclude_unset=True, exclude_none=True)
    updated = stored.model_copy(update=changes)
    items[item_id] = updated
    return updated


@app.delete("/items/{item_id}", status_code=status.HTTP_204_NO_CONTENT)
async def delete_item(item_id: int):
    get_item_or_404(item_id)
    del items[item_id]
    return Response(status_code=status.HTTP_204_NO_CONTENT)
```

真实项目还需要数据库事务、唯一性约束、并发处理、分页上限和权限控制。框架负责把 HTTP 请求映射到 Python 调用，但数据一致性仍由业务逻辑与数据库共同保证。

## 测试与依赖覆盖

FastAPI 的 `TestClient` 可以不启动真实网络端口，直接测试完整的路由、校验、依赖和响应过程。

```python
from fastapi.testclient import TestClient

from main import app

client = TestClient(app)


def test_create_item():
    response = client.post(
        "/items",
        json={"name": "键盘", "price": 499},
    )

    assert response.status_code == 201
    assert response.json()["name"] == "键盘"


def test_rejects_invalid_price():
    response = client.post(
        "/items",
        json={"name": "键盘", "price": 0},
    )

    assert response.status_code == 422
```

测试不应访问真实支付服务或生产数据库。FastAPI 可以临时替换依赖：

```python
from main import User, app, get_current_user


def fake_current_user() -> User:
    return User(id=1, role="admin")


app.dependency_overrides[get_current_user] = fake_current_user
# 执行测试
app.dependency_overrides.clear()
```

实际测试套件通常用 pytest fixture 在每个测试前设置测试数据库和依赖覆盖，并在测试后清理。至少覆盖成功路径、输入校验、资源不存在、权限不足和关键业务冲突。

## 配置管理

端口、数据库地址、第三方密钥等配置会随环境变化，不应散落在代码中。`pydantic-settings` 可以从环境变量和 `.env` 文件读取并校验配置。

```bash
uv add pydantic-settings
```

```python
from functools import lru_cache

from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env",
        env_prefix="APP_",
        extra="ignore",
    )

    environment: str = "development"
    database_url: str
    secret_key: str


@lru_cache
def get_settings() -> Settings:
    return Settings()
```

对应环境变量：

```bash
$env:APP_DATABASE_URL="postgresql://localhost/app"
$env:APP_SECRET_KEY="replace-me"
```

配置对象也可以通过 `Depends(get_settings)` 注入。`.env` 适合本地开发，但包含真实密钥的文件不能提交到 Git；生产环境应由部署平台或密钥管理系统注入。应用启动时就让缺失或格式错误的关键配置报错，比运行到某个请求才失败更容易排查。

## 生产运行边界

开发服务器能运行不等于已经适合生产。生产中的 FastAPI 只是系统的一部分，还要处理进程管理、流量入口、数据库迁移、日志、监控和故障恢复。

```bash
fastapi run app/main.py --port 8000
```

需要在单台机器上使用多个 worker 时可以配置进程数；在容器平台中，常见做法是每个容器运行一个 worker，再通过增加容器副本扩容。具体选择取决于 CPU、内存、连接数和平台的进程管理方式。

上线前至少确认：

| 方面 | 要确认的内容 |
| --- | --- |
| 进程 | 禁用 `--reload`，进程异常后能自动重启 |
| 网络 | HTTPS、反向代理、可信代理头、请求体大小限制 |
| 数据 | 连接池上限、事务边界、迁移流程、备份与恢复 |
| 稳定性 | 外部调用超时、重试边界、限流、优雅关闭 |
| 可观测性 | 结构化日志、请求 ID、指标、链路追踪、错误告警 |
| 安全 | 密钥注入、最小权限、依赖更新、接口认证与授权 |

worker 数量不是越多越好。每个进程都要占内存并建立自己的数据库连接池；盲目增加 worker 可能先耗尽数据库连接或机器内存。应通过压测和生产指标决定并发配置。

## 新手常见误区

| 误区 | 实际情况 |
| --- | --- |
| FastAPI 自己监听端口 | 监听端口的是 Uvicorn 等 ASGI 服务器 |
| 写成 `async def` 就一定更快 | 只有正确等待异步 I/O 才能提高并发；阻塞代码反而会卡住事件循环 |
| 类型注解只是给编辑器看 | FastAPI 会在运行时读取注解，用于解析、校验、序列化和文档 |
| Pydantic 模型可以一套到底 | 创建输入、更新输入、数据库模型和公开响应的字段边界不同 |
| `OAuth2PasswordBearer` 会自动验证 JWT | 它只提取 Bearer 令牌并描述安全方案，令牌验证仍需应用实现 |
| `BackgroundTasks` 是可靠任务队列 | 它在当前进程内执行，进程退出时任务可能丢失 |
| 开放 CORS 等于接口不安全 | CORS 是浏览器策略；真正的安全边界是认证、授权和服务端校验 |
| 多 worker 会共享全局变量 | 每个 worker 是独立进程，内存状态互不共享 |
| 自动文档正确就代表接口设计正确 | 文档只是代码声明的投影，错误的数据模型和状态码会生成错误契约 |

**面试提示**：常见问题集中在 FastAPI、Starlette、Pydantic 和 Uvicorn 的分工，ASGI 与 WSGI 的区别，`async def` 何时有效，依赖注入的执行与清理方式，以及 `response_model` 为什么能防止字段泄露。回答时应从一次请求的完整链路说明，而不是只说“FastAPI 性能高、自动生成文档”。